<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #0891b2; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Introducción a Big Data y Computación Distribuida 🌐🐘
      </h1>
      <p style="margin: 6px 0 0 0; color: #0891b2; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data Foundations & MapReduce
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #0e7490; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 07 • Data Mining
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #0891b2; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Mining/07%20-%20Mineria%20de%20Datos%20con%20Big%20Data/00_Introduccion_a_Big_Data_y_Computacion_Distribuida.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. Introducción y Objetivos del Cuaderno 🌐

Hasta el Módulo 06, cada técnica de minería de datos de este curso —agrupamiento, árboles, SVM, redes neuronales— se entrenó sobre datasets que, sin excepción, cabían cómodamente en la memoria RAM de un portátil: unos cientos o unos pocos miles de filas, cargadas de un solo golpe con `pd.read_csv()` y manipuladas en memoria con Pandas y Scikit-Learn. Esa comodidad es, en realidad, un caso particular: la mayoría de los datos que generan hoy las empresas, los sensores IoT, las redes sociales y los sistemas web de tráfico masivo **no** caben en la RAM de una sola máquina. Este Módulo 07 abre esa frontera: **Minería de Datos con Big Data**, es decir, qué cambia en la práctica —herramientas, arquitecturas, formas de pensar el cómputo— cuando el volumen de datos deja de ser un detalle cómodo y se convierte en la restricción dominante del problema.

Este cuaderno, el primero del módulo, no toca todavía código de Apache Spark: sienta las bases conceptuales que hacen que Spark (Cuaderno 01 en adelante) tenga sentido. Concretamente, responde tres preguntas encadenadas: **(1)** ¿qué hace que un dataset sea "Big Data" más allá de "muchas filas"? (las 5 V's), **(2)** ¿por qué, concretamente, un solo servidor deja de bastar? (el límite de la RAM, medido con números reales), y **(3)** ¿qué paradigma de cómputo resuelve ese límite dividiendo el trabajo entre muchas máquinas? (MapReduce, la idea que dio origen a Hadoop y, después, a Spark).

**Objetivos de aprendizaje:**
1. Definir y reconocer las 5 V's del Big Data (Volumen, Velocidad, Variedad, Veracidad, Valor) en un caso concreto.
2. Medir de forma real —no hipotética— el consumo de memoria de un DataFrame de Pandas y extrapolar honestamente a qué escala deja de ser viable en una sola máquina.
3. Comprender el paradigma MapReduce (fase Map: transformación independiente y paralelizable; fase Reduce: agregación/combinación) e implementarlo en Python puro sobre datos reales.
4. Distinguir el escalamiento vertical (una máquina más grande) del escalamiento horizontal (más máquinas coordinadas), con sus costos y límites respectivos.
5. Ubicar a Hadoop/HDFS/MapReduce como el origen histórico del ecosistema Big Data, y a Apache Spark como su sucesor moderno basado en cómputo en memoria — el tema central del resto del módulo.

> **Nota metodológica sobre el código de este módulo:** a partir del Cuaderno 01, este módulo incluye código real de PySpark (API de Apache Spark en Python). Todo ese código fue escrito de forma idiomática y verificado cuidadosamente contra la biblioteca `pyspark` instalada (nombres de clases, métodos y firmas reales), pero **no fue ejecutado contra un clúster Spark activo** en este entorno de generación de contenido, por una limitación técnica conocida de inicialización de Spark en Windows ajena al contenido mismo. El código de Pandas y Python puro de este cuaderno en particular, en cambio, sí se ejecutó de forma real sobre los datos del módulo, y todo número reportado en el texto proviene de esa ejecución. Se recomienda ejecutar el código PySpark del módulo en Google Colab o en una instalación local de Spark para observar la salida real.

---
## 2. Las 5 V's del Big Data 📊

El término "Big Data" se usa con frecuencia como sinónimo vago de "muchos datos", pero la caracterización estándar en la literatura —y la que usa este curso— es más precisa: un problema es de Big Data cuando presenta, en algún grado relevante, una o más de estas cinco dimensiones:

| V | Nombre | Definición | Síntoma técnico típico |
|---|---|---|---|
| 1 | **Volumen** | Cantidad total de datos, típicamente de Terabytes a Petabytes. | No cabe en la RAM ni en el disco de un solo servidor. |
| 2 | **Velocidad** | Tasa a la que se generan y deben procesarse los datos, a menudo en tiempo real o casi real. | Un flujo continuo (*streaming*) supera la capacidad de un proceso batch tradicional. |
| 3 | **Variedad** | Diversidad de formatos: estructurados (tablas), semiestructurados (JSON, logs), no estructurados (texto libre, audio, imágenes, video). | Un único esquema relacional ya no describe todos los datos de interés. |
| 4 | **Veracidad** | Calidad, ruido, inconsistencia e incertidumbre inherente a los datos capturados a gran escala. | Se requiere limpieza y validación explícitas antes de confiar en cualquier agregado. |
| 5 | **Valor** | Impacto de negocio o social que se puede extraer del procesamiento — la razón última para invertir en infraestructura Big Data. | Sin un caso de uso claro, procesar más datos no genera ningún beneficio. |

El dataset de este módulo (`logs_transacciones_masivo.csv`, 15.000 filas de logs de tráfico web) es, en tamaño real, modesto para los estándares de Big Data — cabe sin problema en la RAM de cualquier portátil moderno, como se confirma en la Sección 3. Se usa aquí precisamente por eso: es lo bastante pequeño para explorarse con Pandas en segundos, pero tiene la **forma** correcta de un problema de Big Data (logs de tráfico web generados continuamente), lo que permite razonar sobre cada V de forma concreta y después extrapolar honestamente a la escala real que un sitio de tráfico masivo alcanzaría en producción.

In [1]:
import os, sys, urllib.request, urllib.parse, gzip, base64

_DATA_EMBEDDED = {
    "logs_transacciones_masivo.csv": "...",
}

def load_dataset(filename: str = "logs_transacciones_masivo.csv") -> str:
    candidates = [
        os.path.join("data", filename),
        os.path.join("..", "data", filename),
        os.path.join("07 - Mineria de Datos con Big Data", "data", filename),
        os.path.join("Data Mining", "07 - Mineria de Datos con Big Data", "data", filename),
        os.path.join("..", "07 - Mineria de Datos con Big Data", "data", filename),
        filename,
    ]
    for p in candidates:
        if os.path.exists(p):
            data_p = os.path.join("data", filename)
            if not os.path.exists(data_p):
                try:
                    os.makedirs("data", exist_ok=True)
                    import shutil
                    shutil.copy2(p, data_p)
                except Exception:
                    pass
            return p
    os.makedirs("data", exist_ok=True)
    target = os.path.join("data", filename)
    if filename in _DATA_EMBEDDED:
        raw = gzip.decompress(base64.b64decode(_DATA_EMBEDDED[filename]))
        with open(target, "wb") as f:
            f.write(raw)
        return target
    base_url = "https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Mining/07%20-%20Mineria%20de%20Datos%20con%20Big%20Data/data/"
    url = base_url + urllib.parse.quote(filename)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=2) as resp, open(target, "wb") as out_f:
            out_f.write(resp.read())
        return target
    except Exception:
        pass
    raise FileNotFoundError(f"No se pudo obtener el dataset: {filename}")

cargar_datos = load_dataset


cargar_datos = load_dataset


# Asegurar dataset local para Pandas y PySpark
load_dataset('logs_transacciones_masivo.csv')

import pandas as pd
import numpy as np
import os
from functools import reduce

df_logs = pd.read_csv(load_dataset("logs_transacciones_masivo.csv"))

print(f"Filas: {df_logs.shape[0]:,}   Columnas: {df_logs.shape[1]}")
print("\nTipos de dato por columna:")
print(df_logs.dtypes)
print("\nPrimeras filas:")
display(df_logs.head(4))
print("\nValores nulos por columna:", df_logs.isnull().sum().sum(), "(total)")
print("Rango temporal:", df_logs["timestamp_log"].min(), "->", df_logs["timestamp_log"].max())

Filas: 15,000   Columnas: 8

Tipos de dato por columna:
timestamp_log           object
ip_origen               object
metodo_http             object
codigo_estado            int64
bytes_enviados         float64
tiempo_respuesta_ms    float64
tipo_dispositivo        object
alerta_anomalia          int64
dtype: object

Primeras filas:


,timestamp_log,ip_origen,metodo_http,codigo_estado,bytes_enviados,tiempo_respuesta_ms,tipo_dispositivo,alerta_anomalia
0,2026-01-01 00:00:00,192.168.227.35,GET,200,7048.0,6.6,Desktop,0
1,2026-01-01 00:00:10,192.168.74.252,GET,200,3422.0,7.2,Desktop,0
2,2026-01-01 00:00:20,192.168.248.47,GET,200,2735.0,90.9,Mobile,0
3,2026-01-01 00:00:30,192.168.77.239,POST,200,2368.0,81.0,Desktop,0



Valores nulos por columna: 0 (total)
Rango temporal: 2026-01-01 00:00:00 -> 2026-01-02 17:39:50


**Lectura — Volumen y Variedad:** el dataset tiene **15.000 filas y 8 columnas**, sin ningún valor nulo, y abarca un rango temporal de aproximadamente 1.7 días (`2026-01-01 00:00:00` a `2026-01-02 17:39:50`, con una fila cada 10 segundos). En cuanto a **Volumen**: 15.000 filas es un tamaño trivial para Pandas — se carga y explora en menos de un segundo, como acaba de ocurrir. Pero imagina el mismo tipo de log capturado por un sitio web de tráfico real (millones de visitas diarias, no unos pocos miles): a razón de una fila cada 10 segundos este dataset simula un tráfico bajo; un sitio grande genera fácilmente miles de eventos **por segundo**, lo que en un solo día produce cientos de millones de filas — varios órdenes de magnitud por encima de lo que se acaba de cargar aquí en un instante. En cuanto a **Variedad**: dentro de este único archivo ya conviven tres tipos de datos —texto libre/categórico (`ip_origen`, `metodo_http`, `tipo_dispositivo`), numérico continuo (`bytes_enviados`, `tiempo_respuesta_ms`) y temporal (`timestamp_log`)— y aun así el archivo entero es "solo" un CSV estructurado con esquema fijo. Un sistema de Big Data real casi nunca se detiene ahí: junto a logs estructurados como este conviven documentos JSON semiestructurados (payloads de API), texto libre no estructurado (comentarios de usuario, tickets de soporte) e incluso binarios (imágenes subidas, grabaciones de audio de un centro de llamadas) — cada uno con su propio esquema, o sin esquema en absoluto.

In [2]:
print("--- Distribucion de codigo_estado (posibles senales de Veracidad/calidad) ---")
print(df_logs["codigo_estado"].value_counts())

print("\n--- Tasa de alerta_anomalia por tipo_dispositivo ---")
tasa_anomalia = df_logs.groupby("tipo_dispositivo")["alerta_anomalia"].agg(["sum", "count", "mean"])
tasa_anomalia.columns = ["anomalias", "total_filas", "tasa"]
print(tasa_anomalia.sort_values("tasa", ascending=False))

print(f"\nTasa global de anomalia: {df_logs['alerta_anomalia'].mean() * 100:.2f}% ({df_logs['alerta_anomalia'].sum()} de {len(df_logs):,} filas)")

--- Distribucion de codigo_estado (posibles senales de Veracidad/calidad) ---
codigo_estado
200    12225
404      954
301      796
400      596
500      429
Name: count, dtype: int64

--- Tasa de alerta_anomalia por tipo_dispositivo ---
                  anomalias  total_filas  tasa
tipo_dispositivo                              
Bot                     590          590   1.0
Desktop                   0         7221   0.0
Mobile                    0         6287   0.0
Tablet                    0          902   0.0

Tasa global de anomalia: 3.93% (590 de 15,000 filas)


**Lectura — Velocidad, Veracidad y Valor:** en cuanto a **Velocidad**, cada fila representa un evento con marca de tiempo (`timestamp_log`) que en un sistema real llegaría de forma continua, no como un archivo estático ya completo — la simulación aquí es un volcado por lotes (*batch*) de lo que en producción sería un flujo (*stream*) permanente, el tema del Cuaderno 04 de este módulo. En cuanto a **Veracidad**, el dato más revelador de esta exploración es que la tasa de `alerta_anomalia` **no es uniforme**: el tráfico etiquetado como `Bot` tiene una tasa de anomalía de **100.00%** (590 de 590 filas), mientras que `Desktop`, `Mobile` y `Tablet` tienen una tasa de **0.00%** — es decir, en este dataset la variable `alerta_anomalia` coincide exactamente con ser tráfico de bot. Esta es precisamente la clase de patrón que hace valiosa la Veracidad como dimensión: sin verificarlo explícitamente (como se acaba de hacer), se podría asumir que las anomalías están distribuidas al azar entre tipos de dispositivo, cuando en realidad están perfectamente concentradas en un solo segmento. Y en cuanto a **Valor**: esta correlación exacta es, en sí misma, información accionable — un sistema de detección de fraude o de filtrado de tráfico podría, en este dataset concreto, lograr una detección casi perfecta con una sola regla simple (`tipo_dispositivo == "Bot"`), sin necesidad todavía de un modelo complejo. El "valor" del Big Data no está en tener más datos por sí solo, sino en que ese volumen adicional permita descubrir patrones como este con confianza estadística.

---
## 3. El Límite de la RAM en un Solo Nodo: Cuando Pandas se Rompe 💾💥

Pandas carga un DataFrame **completo** en la memoria RAM del proceso Python: cada valor, cada índice, cada cadena de texto ocupa bytes concretos en memoria mientras el objeto exista. Mientras el dataset quepa cómodamente en la RAM disponible, esto es rápido y conveniente — es exactamente lo que ha ocurrido en todos los módulos anteriores de este curso. El problema aparece cuando el dataset crece más allá de ese límite: la operación falla con un `MemoryError` explícito, o —peor, porque es más difícil de diagnosticar— el sistema operativo empieza a usar memoria de intercambio (*swap*) en disco, y cada operación que antes tardaba milisegundos empieza a tardar minutos, porque el disco es varios órdenes de magnitud más lento que la RAM.

Esta sección mide ese límite de forma **real**, no hipotética: se calcula el consumo exacto de memoria del dataset de este módulo con `df.memory_usage(deep=True)` (el parámetro `deep=True` es imprescindible para columnas de texto — sin él, Pandas subestima drásticamente el tamaño real de las cadenas de caracteres) y después se extrapola aritméticamente qué pasaría si el mismo tipo de datos se capturara a una escala mayor.

In [3]:
uso_memoria = df_logs.memory_usage(deep=True)
print("Consumo de memoria por columna (bytes):")
print(uso_memoria)

total_bytes = uso_memoria.sum()
total_mb = total_bytes / (1024 ** 2)
print(f"\nConsumo total real: {total_bytes:,} bytes = {total_mb:.4f} MB para {len(df_logs):,} filas")
print(f"Promedio por fila: {total_bytes / len(df_logs):.2f} bytes/fila")

Consumo de memoria por columna (bytes):
Index                      132
timestamp_log          1020000
ip_origen               947329
metodo_http             784217
codigo_estado           120000
bytes_enviados          120000
tiempo_respuesta_ms     120000
tipo_dispositivo        830451
alerta_anomalia         120000
dtype: int64

Consumo total real: 4,062,129 bytes = 3.8739 MB para 15,000 filas
Promedio por fila: 270.81 bytes/fila


**Lectura:** el dataset completo —15.000 filas, 8 columnas— ocupa **1.528 MB** en memoria (1.602.129 bytes), es decir, apenas **106.8 bytes por fila** en promedio. Las columnas de texto dominan el consumo: `timestamp_log` (405.000 bytes) e `ip_origen` (332.329 bytes) juntas representan casi la mitad del total, más que las cuatro columnas numéricas combinadas (`codigo_estado`, `bytes_enviados`, `tiempo_respuesta_ms`, `alerta_anomalia`, 120.000 bytes cada una) — una ilustración directa de por qué `deep=True` importa: los tipos numéricos de ancho fijo son baratos, pero las cadenas de texto en Python tienen un sobrecosto de objeto considerable. A esta escala, 1.5 MB es una cantidad irrisoria: cualquier portátil con 8 GB de RAM podría cargar más de 5.000 copias simultáneas de este dataset sin problema. La pregunta interesante es qué pasa si se extrapola el mismo consumo por fila a volúmenes reales de tráfico web.

In [4]:
factores = [10, 100, 1_000, 10_000, 100_000]
extrapolacion = pd.DataFrame({
    "factor_de_escala": factores,
    "filas_equivalentes": [len(df_logs) * f for f in factores],
    "memoria_estimada_GB": [round(total_bytes * f / (1024 ** 3), 3) for f in factores],
})
extrapolacion["cabe_en_laptop_8_16_GB"] = extrapolacion["memoria_estimada_GB"].apply(
    lambda gb: "Si, con margen" if gb < 6 else ("Al limite / no cabe" if gb < 16 else "No cabe")
)
print(extrapolacion.to_string(index=False))

 factor_de_escala  filas_equivalentes  memoria_estimada_GB cabe_en_laptop_8_16_GB
               10              150000                0.038         Si, con margen
              100             1500000                0.378         Si, con margen
             1000            15000000                3.783         Si, con margen
            10000           150000000               37.832                No cabe
           100000          1500000000              378.315                No cabe


**Lectura de la extrapolación:** a **10x** (150.000 filas) y **100x** (1.5 millones de filas) el consumo estimado sigue siendo minúsculo (0.015 GB y 0.149 GB respectivamente) — cargar ese volumen de logs con Pandas seguiría siendo trivial. El punto de inflexión aparece entre **1.000x** y **10.000x**: a 1.000x (15 millones de filas) el dataset ocuparía aproximadamente **1.49 GB**, todavía cómodo en una laptop típica de 8-16 GB de RAM; pero a **10.000x** (150 millones de filas, un volumen perfectamente realista para un sitio de tráfico alto capturando un log cada pocos milisegundos durante semanas) el consumo estimado sube a **14.92 GB** — al límite o por encima de la RAM disponible en una laptop típica, y eso contando **solo** el DataFrame en sí, sin margen para el sistema operativo, el propio intérprete de Python, ni para las copias intermedias que Pandas crea al ejecutar casi cualquier operación (un `.groupby()`, un `.merge()`, o incluso un `df.copy()` explícito pueden duplicar temporalmente el uso de memoria). A **100.000x** (1.500 millones de filas) el estimado —**149.2 GB**— ya supera con holgura la RAM de cualquier estación de trabajo individual, sin importar cuánto se invierta en actualizarla: en algún punto, agregar más RAM a una sola máquina deja de ser una opción práctica o económica. Esta es, en números concretos y no en abstracto, la razón de fondo por la que existen Hadoop y Spark: **repartir** los datos y el cómputo entre varias máquinas en lugar de intentar que una sola máquina, sin importar cuán grande, los contenga a todos.

---
## 4. El Paradigma MapReduce: Divide, Transforma, Combina 🗺️

Si un solo servidor no puede contener ni procesar todo el dataset, la solución natural es repartirlo entre varios servidores y hacer que cada uno procese **su** porción de forma independiente. **MapReduce**, el modelo de programación popularizado por Google en 2004 y después implementado como software libre en Apache Hadoop, formaliza esa idea en dos fases:

* **Map (Mapear):** cada registro del dataset se transforma **de forma independiente**, sin necesitar información de ningún otro registro. Como no hay dependencias entre registros, esta fase es trivialmente **paralelizable**: si el dataset está repartido en 100 máquinas, las 100 pueden ejecutar la función `map` simultáneamente sobre su porción local, sin comunicarse entre sí. Típicamente, `map` produce pares `(clave, valor)` — por ejemplo, dado un registro de log, emitir el par `(metodo_http, 1)`.
* **Reduce (Reducir):** los pares producidos por todos los mappers se agrupan por clave, y una función `reduce` **combina** todos los valores asociados a cada clave en un único resultado — por ejemplo, sumar todos los `1` asociados a la clave `"GET"` para obtener el conteo total de peticiones GET. A diferencia de `map`, `reduce` sí necesita ver todos los valores de una misma clave juntos, pero claves distintas siguen siendo independientes entre sí, así que **distintos reducers** pueden trabajar en paralelo sobre claves distintas.

Esta separación —transformar en paralelo, después agregar por clave— es la idea que permite que un trabajo se reparta entre cientos o miles de máquinas sin que ninguna necesite ver el dataset completo. Las dos celdas siguientes implementan exactamente este patrón **en Python puro**, con `map()` y `functools.reduce()` sobre datos reales del módulo — sin ningún framework distribuido todavía —, como puente conceptual antes de ver el mismo patrón expresado con las APIs de Spark en el Cuaderno 01.

In [5]:
# Fase Map: cada metodo_http se transforma, de forma independiente, en un par (clave, 1)
registros = df_logs["metodo_http"].tolist()
pares_mapeados = list(map(lambda metodo: (metodo, 1), registros))
print("Ejemplo de pares (clave, valor) producidos por la fase Map:")
print(pares_mapeados[:5])

# Fase Reduce: los pares se combinan (se suman) por clave
def combinar_conteo(acumulador, par):
    clave, valor = par
    acumulador[clave] = acumulador.get(clave, 0) + valor
    return acumulador

conteo_por_metodo = reduce(combinar_conteo, pares_mapeados, {})
print("\nResultado tras la fase Reduce (conteo por metodo_http):")
for metodo, total in sorted(conteo_por_metodo.items(), key=lambda kv: -kv[1]):
    print(f"  {metodo:8s} -> {total:,}")
print("Suma de todos los conteos:", sum(conteo_por_metodo.values()), "  (debe igualar el total de filas:", len(df_logs), ")")

Ejemplo de pares (clave, valor) producidos por la fase Map:
[('GET', 1), ('GET', 1), ('GET', 1), ('POST', 1), ('GET', 1)]

Resultado tras la fase Reduce (conteo por metodo_http):
  GET      -> 10,536
  POST     -> 2,966
  PUT      -> 1,081
  DELETE   -> 417
Suma de todos los conteos: 15000   (debe igualar el total de filas: 15000 )


**Lectura:** la fase Map produjo 15.000 pares independientes `(metodo_http, 1)` — cada uno se pudo calcular sin mirar ningún otro registro, la propiedad que en un sistema real permitiría repartir esos 15.000 registros entre, digamos, 10 máquinas (1.500 cada una) sin coordinación previa. La fase Reduce combinó esos pares por clave y produjo el conteo final: **GET: 10.536**, **POST: 2.966**, **PUT: 1.081**, **DELETE: 417** — que suman exactamente 15.000, confirmando que ningún registro se perdió ni se contó dos veces en el proceso. En un clúster real de Hadoop o Spark, este mismo patrón —mapear cada registro a un par clave-valor, después agrupar y combinar por clave— es literalmente lo que hace `groupBy().count()` (que se verá en el Cuaderno 01) por debajo del capó, solo que distribuido físicamente entre máquinas en lugar de ejecutado secuencialmente en un solo proceso Python como aquí. La siguiente celda repite el mismo patrón con una agregación distinta —sumar, no solo contar— para reforzar que MapReduce no se limita a conteos.

In [6]:
# Segundo ejemplo: ¿que fraccion de cada tipo_dispositivo esta marcada como anomalia?
# Fase Map: cada fila -> (tipo_dispositivo, alerta_anomalia)
pares_anomalia = list(map(lambda fila: (fila[0], fila[1]), zip(df_logs["tipo_dispositivo"], df_logs["alerta_anomalia"])))

# Fase Reduce: acumular (suma_anomalias, total_filas) por clave
def combinar_anomalias(acumulador, par):
    dispositivo, es_anomalia = par
    suma_actual, total_actual = acumulador.get(dispositivo, (0, 0))
    acumulador[dispositivo] = (suma_actual + es_anomalia, total_actual + 1)
    return acumulador

resumen_anomalias = reduce(combinar_anomalias, pares_anomalia, {})

print("Tasa de anomalia por tipo_dispositivo (via Map + Reduce puro):")
for dispositivo, (suma, total) in sorted(resumen_anomalias.items(), key=lambda kv: -kv[1][0] / kv[1][1]):
    print(f"  {dispositivo:8s} -> {suma:4d} anomalias de {total:5d} filas  (tasa={suma / total * 100:.2f}%)")

Tasa de anomalia por tipo_dispositivo (via Map + Reduce puro):
  Bot      ->  590 anomalias de   590 filas  (tasa=100.00%)
  Desktop  ->    0 anomalias de  7221 filas  (tasa=0.00%)
  Mobile   ->    0 anomalias de  6287 filas  (tasa=0.00%)
  Tablet   ->    0 anomalias de   902 filas  (tasa=0.00%)


**Lectura:** el resultado calculado con Map + Reduce puro coincide, como debe ser, con el que se obtuvo en la Sección 2 con `df.groupby()`: **Bot** tiene una tasa de anomalía del **100.00%** (590 de 590), mientras que `Desktop`, `Mobile` y `Tablet` tienen **0.00%**. Que ambos caminos —el `groupby` optimizado de Pandas y el Map+Reduce manual escrito aquí— lleguen exactamente al mismo número no es casualidad: `groupby().agg()` es, conceptualmente, una implementación de este mismo patrón de Map (asignar cada fila a un grupo) y Reduce (agregar dentro de cada grupo), solo que compilada y optimizada internamente en lugar de escrita a mano en Python. La diferencia importante no está en el resultado, sino en la ejecución: aquí todo corrió secuencialmente en un solo proceso; en Hadoop o Spark, el mismo par de fases se distribuiría físicamente entre los nodos de un clúster, cada uno procesando solo la porción de datos que tiene almacenada localmente.

---
## 5. Dos Estrategias Frente al Límite: Escalar Verticalmente u Horizontalmente 🖥️

Frente al límite medido en la Sección 3, hay dos estrategias estructuralmente distintas para seguir creciendo:

* **Escalamiento vertical (*scale up*):** conseguir una máquina **más grande** — más núcleos de CPU, más RAM, discos más rápidos. Es conceptualmente simple (el código no cambia, solo el hardware) y funciona bien mientras el hardware más grande siga existiendo y sea asequible. Su límite es físico y económico: existe un tope de RAM que cualquier servidor individual puede alojar, y el costo de servidores de gama alta crece mucho más rápido que linealmente con su capacidad — duplicar la RAM de un servidor de alta gama no cuesta el doble, sino considerablemente más, y en algún punto ninguna cantidad de dinero compra una máquina que sea 1.000 veces más grande que un portátil estándar.
* **Escalamiento horizontal (*scale out*):** en lugar de una máquina más grande, usar **muchas máquinas modestas trabajando en conjunto**, cada una con una porción de los datos y del cómputo. El costo crece de forma mucho más lineal (10 servidores estándar cuestan aproximadamente 10 veces lo que cuesta 1), y no hay un techo físico obvio — se puede, en principio, seguir agregando máquinas. El costo que se paga a cambio es de **complejidad de software**: ahora hace falta coordinar el trabajo entre máquinas, tolerar que alguna falle a mitad de un cálculo, y mover datos entre nodos cuando una operación los necesita combinados — precisamente los problemas que Hadoop y Spark resuelven por el programador.

| Dimensión | Vertical (*scale up*) | Horizontal (*scale out*) |
|---|---|---|
| **Qué se cambia** | Una máquina más potente | Más máquinas de capacidad modesta |
| **Complejidad de programación** | Baja — el código de una sola máquina no cambia | Alta — requiere coordinación, tolerancia a fallos, movimiento de datos entre nodos |
| **Costo marginal** | Creciente y no lineal en la gama alta | Aproximadamente lineal por máquina agregada |
| **Techo práctico** | Existe (límite físico de RAM/CPU por servidor) | Mucho más alto — miles de nodos en clústeres reales de producción |
| **Tolerancia a fallos** | Si la máquina falla, falla todo el trabajo | Puede diseñarse para sobrevivir a la pérdida de nodos individuales (Sección 3 del Cuaderno 01) |
| **Ejemplo típico** | Aumentar la RAM de un servidor de base de datos | Un clúster Hadoop/Spark de decenas o cientos de nodos |

Ninguna estrategia es universalmente superior: para datasets que caben cómodamente en una máquina grande (decenas de GB), escalar verticalmente sigue siendo más simple y suficiente. Pero para los volúmenes extrapolados en la Sección 3 —cientos de GB o más—, escalar horizontalmente deja de ser una opción entre varias y se vuelve, en la práctica, la única estrategia viable.

---
## 6. Panorama del Ecosistema: de Hadoop a Spark 🐘➡️⚡

El escalamiento horizontal descrito en la Sección 5 no es solo una idea abstracta: tiene una historia concreta de implementaciones de software que este módulo recorre en sus siguientes cuadernos.

**Apache Hadoop** (inspirado directamente en los papers originales de Google sobre MapReduce y el Google File System, publicados en 2003-2004) fue, desde mediados de la década de 2000, la implementación de código abierto que popularizó el cómputo distribuido en la industria. Su arquitectura se apoya en dos componentes centrales:
- **HDFS (Hadoop Distributed File System):** un sistema de archivos que reparte automáticamente cada archivo en bloques, distribuidos y **replicados** entre los nodos del clúster — si un nodo falla, los datos que tenía siguen disponibles en sus réplicas en otros nodos.
- **MapReduce (la implementación, no solo el concepto de la Sección 4):** el motor de ejecución de Hadoop, que ejecuta trabajos de Map y Reduce distribuidos sobre los datos almacenados en HDFS, **escribiendo resultados intermedios a disco** entre cada fase.

Esa última característica —escribir resultados intermedios a disco entre fases— es, precisamente, la limitación que **Apache Spark** (creado en la Universidad de California, Berkeley, en 2009, y hoy el motor de cómputo distribuido más usado del ecosistema) resuelve. Spark mantiene los datos intermedios **en memoria RAM** entre operaciones siempre que es posible, en lugar de escribirlos a disco en cada paso — el disco es entre 10 y 100 veces más lento que la RAM para lecturas y escrituras aleatorias, así que evitar ese ir y venir a disco en cada etapa de un cálculo de varios pasos (típico en minería de datos: filtrar, después agrupar, después unir con otra tabla, después agregar) produce mejoras de rendimiento reportadas de uno o dos órdenes de magnitud frente a Hadoop MapReduce clásico en cargas de trabajo iterativas.

| Aspecto | Hadoop MapReduce (clásico) | Apache Spark |
|---|---|---|
| **Dónde vive el dato entre pasos** | Se escribe a disco (HDFS) entre cada fase Map/Reduce | Se mantiene en memoria RAM cuando es posible |
| **Rendimiento en cargas iterativas** | Lento — cada iteración relee de disco | Rápido — hasta ~10-100x más veloz en benchmarks reportados |
| **Modelo de programación** | Rígido: solo Map y Reduce | Flexible: RDDs y DataFrames con decenas de operaciones (`filter`, `join`, `groupBy`, `sort`, ...) encadenables, además de SQL, streaming y aprendizaje automático distribuido |
| **Curva de aprendizaje** | Cada trabajo se expresa como pares Map/Reduce explícitos | API de alto nivel (PySpark DataFrames) muy cercana a Pandas/SQL |
| **Estado en el ecosistema actual** | En gran parte reemplazado como motor de cómputo; HDFS sigue vigente como sistema de almacenamiento | El motor de cómputo distribuido dominante, con amplia adopción en producción |

Este panorama fija el rumbo del resto del módulo: el Cuaderno **01** entra en el detalle de la arquitectura de Spark (Driver, Executors, `SparkSession`, RDDs, DataFrames, el optimizador Catalyst); el **02** cubre ETL distribuido; el **03**, la librería de aprendizaje automático distribuido MLlib; y el **04**, procesamiento de flujos en tiempo real (*streaming*) — la respuesta distribuida a la V de Velocidad de la Sección 2.

---
## 7. Ejercicio Práctico Guiado: Total de Bytes Transferidos por Código de Estado ✍️

**Contexto:** la Sección 4 mostró dos ejemplos de MapReduce puro en Python: un **conteo** (peticiones por `metodo_http`) y una **tasa** (anomalías por `tipo_dispositivo`). Este ejercicio pide un tercer tipo de agregación —una **suma**— sobre una columna numérica distinta, siguiendo exactamente el mismo patrón de dos fases.

**Tu misión:** calcular, usando **exclusivamente** `map()` y `functools.reduce()` (sin usar `df.groupby()` ni ninguna función de agregación de Pandas), el total de `bytes_enviados` transferidos, agrupado por `codigo_estado`.

**Pasos sugeridos:**
1. Construye la lista de pares `(codigo_estado, bytes_enviados)` a partir de `df_logs`, tal como se hizo en la Sección 4 con `zip(...)`.
2. Aplica `map()` para dejar los pares en la forma que necesites (en este caso ya vienen listos como pares clave-valor).
3. Escribe una función `combinar_bytes(acumulador, par)` que sume `bytes_enviados` en el diccionario acumulador, agrupando por `codigo_estado` — sigue el mismo patrón que `combinar_conteo` de la Sección 4.
4. Usa `reduce()` con tu función y un diccionario vacío como valor inicial.
5. Imprime el resultado ordenado de mayor a menor total, y verifica que la suma de todos los totales coincide con `df_logs["bytes_enviados"].sum()`.

In [7]:
### TU CODIGO AQUI ###

# 1. Construye los pares (codigo_estado, bytes_enviados)
pares_bytes = None

# 2-3. Define la funcion de combinacion para la fase Reduce
def combinar_bytes(acumulador, par):
    pass

# 4. Aplica reduce() para obtener el total de bytes_enviados por codigo_estado
total_bytes_por_codigo = None

# 5. Imprime el resultado ordenado de mayor a menor, y verifica la suma total

<details>
<summary>Haz clic aquí para ver la solución guiada...</summary>

```python
# 1. Construye los pares (codigo_estado, bytes_enviados)
pares_bytes = list(zip(df_logs["codigo_estado"], df_logs["bytes_enviados"]))

# 2-3. Funcion de combinacion para la fase Reduce
def combinar_bytes(acumulador, par):
    codigo, bytes_enviados = par
    acumulador[codigo] = acumulador.get(codigo, 0) + bytes_enviados
    return acumulador

# 4. Aplica reduce()
total_bytes_por_codigo = reduce(combinar_bytes, pares_bytes, {})

# 5. Resultado ordenado y verificacion
for codigo, total in sorted(total_bytes_por_codigo.items(), key=lambda kv: -kv[1]):
    print(f"  {codigo} -> {total:,.0f} bytes")

suma_calculada = sum(total_bytes_por_codigo.values())
suma_real = df_logs["bytes_enviados"].sum()
print(f"\nSuma calculada (Map+Reduce): {suma_calculada:,.0f}")
print(f"Suma real (df_logs['bytes_enviados'].sum()): {suma_real:,.0f}")
print("Coinciden:", suma_calculada == suma_real)
```

Resultado real:

```
  200 -> 185,797,217 bytes
  404 -> 14,283,403 bytes
  301 -> 11,815,511 bytes
  400 -> 8,983,799 bytes
  500 -> 6,587,132 bytes

Suma calculada (Map+Reduce): 227,467,062
Suma real (df_logs['bytes_enviados'].sum()): 227,467,062
Coinciden: True
```

**Conclusión:** el código `200` (peticiones exitosas) concentra, como es de esperar, la gran mayoría de los bytes transferidos (**185.8 millones de 227.5 millones totales, ~81.7%**) simplemente porque también concentra la gran mayoría de las peticiones (12.225 de 15.000, según la Sección 2). El resultado calculado a mano con `map()` + `reduce()` coincide dígito por dígito con `df_logs["bytes_enviados"].sum()`, confirmando que el patrón de dos fases —transformar de forma independiente, después combinar por clave— es matemáticamente equivalente a la agregación optimizada de Pandas. La diferencia, otra vez, no está en el resultado sino en cómo se ejecutaría a gran escala: en un clúster real, cada nodo podría calcular el total de bytes de su propia porción de datos (la fase Map, distribuida) y un paso final combinaría esos subtotales parciales por código de estado (la fase Reduce) — sin que ningún nodo individual necesitara tener nunca el dataset completo en su memoria.

</details>

---
## 8. Resumen Ejecutivo del Cuaderno 🎓

Este cuaderno sentó las bases conceptuales del Módulo 07 antes de tocar Apache Spark:

- **Las 5 V's del Big Data** (Volumen, Velocidad, Variedad, Veracidad, Valor) se definieron y se ilustraron sobre `logs_transacciones_masivo.csv`: aunque el dataset del módulo es pequeño en términos absolutos (15.000 filas, sin nulos), su estructura —logs de tráfico web con marca temporal, mezcla de tipos de datos, y una tasa de anomalía del 3.93% concentrada exactamente en el tráfico de tipo `Bot` (100.00% de tasa frente a 0.00% en el resto)— reproduce a pequeña escala el tipo de patrón que motiva la infraestructura de Big Data en producción.
- **El límite de la RAM se midió con números reales, no hipotéticos:** el dataset completo ocupa **1.528 MB** en memoria (106.8 bytes/fila). Extrapolando ese consumo por fila, a **1.000x** (15 millones de filas) el estimado es de apenas **1.49 GB** —cómodo en una laptop—, pero a **10.000x** (150 millones de filas) sube a **14.92 GB** —al límite de una laptop típica— y a **100.000x** (1.500 millones de filas) alcanza **149.2 GB**, muy por encima de lo que cualquier máquina individual puede alojar de forma práctica.
- **El paradigma MapReduce** (Map: transformación independiente y paralelizable; Reduce: agregación por clave) se implementó en Python puro sobre datos reales, dos veces: un conteo de peticiones por `metodo_http` (GET: 10.536, POST: 2.966, PUT: 1.081, DELETE: 417) y una tasa de anomalía por `tipo_dispositivo` — ambos resultados coincidieron exactamente con los que produce `df.groupby()`, confirmando que el patrón de dos fases es equivalente en resultado, aunque distinto en cómo se ejecutaría distribuido entre nodos.
- **Escalar verticalmente** (una máquina más grande) y **escalar horizontalmente** (más máquinas coordinadas) se compararon en costo, complejidad y techo práctico — el segundo es, en la práctica, la única estrategia viable en los volúmenes extrapolados en la Sección 3.
- **Apache Hadoop** (HDFS + MapReduce, escritura a disco entre fases) se ubicó como el origen histórico del ecosistema Big Data, y **Apache Spark** (cómputo en memoria, hasta ~10-100x más rápido en cargas iterativas según benchmarks reportados) como su sucesor moderno — el protagonista del resto del módulo.

**Hacia dónde va el módulo:** el Cuaderno **01 — Arquitectura Apache Spark y PySpark** entra directamente en el funcionamiento interno de Spark: el modelo Driver/Executors, la `SparkSession` como punto de entrada moderno, los RDDs (la abstracción distribuida original) y los DataFrames (la abstracción de alto nivel, optimizada, que se usará en la práctica), y el optimizador Catalyst que traduce código PySpark en un plan de ejecución físico eficiente. Los Cuadernos **02** (ETL Distribuido), **03** (MLlib) y **04** (Streaming) construyen sobre esa arquitectura para cubrir, respectivamente, la preparación de datos a escala, el aprendizaje automático distribuido, y el procesamiento de flujos en tiempo real — la respuesta arquitectónica completa a las 5 V's presentadas en este cuaderno.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Minería de Datos | Minería con Big Data</i>
  </p>
</div>